# Deep Hedging — Phase 4, brique 3 : le couvreur neuronal sous Heston

On entraîne le réseau à couvrir un call sur des trajectoires **Heston**, avec un **état augmenté** : en plus de `(log-moneyness, tau, position courante)`, on lui donne la **variance instantanée** `v_t`. C'est cette information de volatilité qui va lui permettre de battre le delta de Black-Scholes, qui l'ignore.

Comme on ne peut pas vectoriser Heston, on **génère la trajectoire pas à pas dans la même boucle** que le déroulé de couverture.

**Cible** (brique 2, sans coûts, n=63) : le delta BS plafonne à CVaR $\approx 4.6$ (le plancher de vega). Le réseau devrait passer nettement dessous.

*Note d'honnêteté : ici on suppose la variance `v_t` observable. En réalité elle est latente (on l'inférerait de la vol réalisée ou des prix d'options) ; c'est le cas idéalisé qui montre que le réseau sait exploiter l'info de vol.*

In [ ]:
import torch
import numpy as np
torch.manual_seed(0)
print("torch", torch.__version__)

## Paramètres, prime, et outils

In [ ]:
S0, K, mu, r, T = 100., 100., 0.05, 0.02, 1.0
v0, kappa, theta, xi, rho = 0.04, 2.0, 0.04, 0.3, -0.7   # vol ~20%
n, alpha = 63, 0.95
dt = T / n

def heston_step(S, v, drift, Z1, Z2):
    """un pas d'Euler Heston (troncature complète), en torch."""
    vk = torch.clamp(v, min=0.0)
    S2 = S * torch.exp((drift - 0.5*vk)*dt + torch.sqrt(vk)*np.sqrt(dt)*Z1)
    v2 = torch.clamp(v + kappa*(theta - vk)*dt + xi*torch.sqrt(vk)*np.sqrt(dt)*Z2, min=0.0)
    return S2, v2

# prime = prix Heston par Monte-Carlo risque-neutre (drift r), sans gradient
with torch.no_grad():
    m0 = 200_000
    S = torch.full((m0,), S0); v = torch.full((m0,), v0)
    for k in range(n):
        Z1 = torch.randn(m0); Z2 = rho*Z1 + np.sqrt(1-rho**2)*torch.randn(m0)
        S, v = heston_step(S, v, r, Z1, Z2)
    premium = float(np.exp(-r*T) * torch.clamp(S-K, min=0.0).mean())
print(f"prime = prix Heston (MC RN) = {premium:.3f}")

def cvar_ru(loss, w, alpha=0.95):
    return w + torch.mean(torch.relu(loss - w)) / (1.0 - alpha)

def bs_delta_torch(S, K, tau, r, s):
    d1 = (torch.log(S/K) + (r + 0.5*s**2)*tau) / (s*np.sqrt(tau))
    return 0.5*(1.0 + torch.erf(d1/np.sqrt(2.0)))

## Le réseau (état à 4 entrées) et le déroulé fusionné sim+couverture

In [ ]:
class HedgeNet(torch.nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(4, hidden), torch.nn.ReLU(),
            torch.nn.Linear(hidden, hidden), torch.nn.ReLU(),
            torch.nn.Linear(hidden, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

def hedging_loss(net, w, m):
    S = torch.full((m,), S0); v = torch.full((m,), v0)
    cash = torch.full((m,), premium); delta_prev = torch.zeros(m)
    for k in range(n):
        tau = float(T - k*dt)
        feat = torch.stack([torch.log(S/K), torch.full((m,), tau), delta_prev, v], dim=1)  # (m,4)
        delta_k = net(feat)
        trade = delta_k - delta_prev
        cash = cash - trade*S                       # sans coûts
        # le marché avance d'un pas (drift physique mu)
        Z1 = torch.randn(m); Z2 = rho*Z1 + np.sqrt(1-rho**2)*torch.randn(m)
        S, v = heston_step(S, v, mu, Z1, Z2)
        cash = cash * np.exp(r*dt)
        delta_prev = delta_k
    pnl = cash + delta_prev*S - torch.clamp(S - K, min=0.0)
    return cvar_ru(-pnl, w, alpha)

## Entraînement

Un peu plus lourd que le GBM (boucle Heston), donc batch modeste. Sur CPU, quelques minutes.

In [ ]:
net = HedgeNet()
w = torch.zeros(1, requires_grad=True)
opt = torch.optim.Adam(list(net.parameters()) + [w], lr=1e-3)

for it in range(2500):
    opt.zero_grad()
    loss = hedging_loss(net, w, 1024)
    loss.backward()
    opt.step()
    if it % 250 == 0:
        print(f"iter {it:4d}   CVaR (train) = {loss.item():.3f}")

## Évaluation : réseau vs delta BS, à la même fréquence

In [ ]:
sig_imp = 0.1942   # vol implicite BS du prix Heston (calculée brique 2)

with torch.no_grad():
    m = 100_000
    # --- réseau ---
    S = torch.full((m,), S0); v = torch.full((m,), v0)
    cash = torch.full((m,), premium); dprev = torch.zeros(m)
    for k in range(n):
        tau = float(T - k*dt)
        feat = torch.stack([torch.log(S/K), torch.full((m,), tau), dprev, v], dim=1)
        dk = net(feat); cash = cash - (dk-dprev)*S
        Z1 = torch.randn(m); Z2 = rho*Z1 + np.sqrt(1-rho**2)*torch.randn(m)
        S, v = heston_step(S, v, mu, Z1, Z2); cash = cash*np.exp(r*dt); dprev = dk
    pnl_net = cash + dprev*S - torch.clamp(S-K, min=0.0)

    # --- delta BS (benchmark), mêmes trajectoires impossibles a rejouer, on resimule ---
    S = torch.full((m,), S0); v = torch.full((m,), v0)
    cash = torch.full((m,), premium); dprev = torch.zeros(m)
    for k in range(n):
        tau = float(T - k*dt)
        dk = bs_delta_torch(S, K, tau, r, sig_imp); cash = cash - (dk-dprev)*S
        Z1 = torch.randn(m); Z2 = rho*Z1 + np.sqrt(1-rho**2)*torch.randn(m)
        S, v = heston_step(S, v, mu, Z1, Z2); cash = cash*np.exp(r*dt); dprev = dk
    pnl_bs = cash + dprev*S - torch.clamp(S-K, min=0.0)

def cvar(pnl, a=0.95):
    loss = -pnl; return loss[loss >= torch.quantile(loss, a)].mean().item()

print(f"CVaR réseau   = {cvar(pnl_net):.3f}")
print(f"CVaR delta BS = {cvar(pnl_bs):.3f}   (benchmark ~4.6)")

## Ce qu'on regardera

Si le réseau passe nettement sous le delta BS, la démonstration de la phase 4 est faite : en marché incomplet, le couvreur appris exploite l'information de volatilité que le delta ignore, et gagne bien plus qu'en GBM. On analysera ensuite *comment* il s'écarte du delta (sa position en fonction de `v`, la correction de type minimum-variance delta). Reste que le vega ne peut être totalement couvert qu'avec un second instrument : ce sera l'extension suivante.